In [1]:
import numpy as np
import copy

np.set_printoptions(suppress=True)  # 禁用科学计数
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
path = "data/西瓜数据集3.1.txt"
data = pd.read_csv(
    path,
    )
data.shape

(17, 8)

In [3]:
data.head()

,Burgundy,slap,texture,umbilicus,touch,density,Sugar content,Good melon
0,青绿,蜷缩,浊响,清晰,凹陷,硬滑,0.697,是
1,乌黑,蜷缩,沉闷,清晰,凹陷,硬滑,0.774,是
2,乌黑,蜷缩,浊响,清晰,凹陷,硬滑,0.634,是
3,青绿,蜷缩,沉闷,清晰,凹陷,硬滑,0.608,是
4,浅白,蜷缩,浊响,清晰,凹陷,硬滑,0.556,是


In [4]:
# 数据是离散的, 要用one-hot处理.
from sklearn import preprocessing
enc = preprocessing.OneHotEncoder()

In [5]:
a = np.array(enc.fit_transform(data.iloc[:, :6]).toarray())
b = np.array(data.iloc[:, 6:8])

X = np.c_[a, b]
y = np.array(enc.fit_transform(data.iloc[:, 8:]).toarray())

# 神经网络
m = log2(n)
【m 为隐层节点数, n 为输入层节点数】

In [7]:
hidden_size = 5
input_size = X.shape[1]
num_labels = y.shape[1]

In [8]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [9]:
def separate_params(params, hidden_size, input_size, num_labels):
    theta1 = np.reshape(params[:hidden_size * (input_size + 1)], (hidden_size, (input_size + 1)))
    theta2 = np.reshape(params[hidden_size * (input_size + 1):], (num_labels, (hidden_size + 1)))
    return [theta1, theta2]

# 前向传播

In [11]:
def forward_propagate(params, X, hidden_size, input_size, num_labels):
    theta = separate_params(params, hidden_size, input_size, num_labels)
    n = X.shape[0]
    ones = np.ones([n, 1])
    a_0 = np.c_[X, ones]  # 加全1列表示偏置项
    z_1 = a_0 @ theta[0].T
    a_1 = np.c_[sigmoid(z_1), ones]  # 同上
    z_2 = a_1 @ theta[1].T
    res = sigmoid(z_2)
    return a_0, z_1, a_1, z_2, res

In [12]:
def create_params(hidden_size, input_size, num_labels):
    return (np.random.random(size=hidden_size * (input_size + 1) + num_labels * (hidden_size + 1)) - 0.5) * 0.25
    # 减0.5的用处是为了让随机值有正有负

In [14]:
params = create_params(hidden_size, input_size, num_labels)
params

array([ 0.04858519,  0.03756161, -0.06863041, -0.0032519 , -0.08901954,
       -0.08896357, -0.00663822, -0.11530095,  0.04682513, -0.12442459,
       -0.0871502 ,  0.12169683,  0.09545919, -0.10012318, -0.09601043,
        0.10515283,  0.11691384,  0.00493814, -0.02652192, -0.04317681,
       -0.11951827, -0.02521176,  0.1087548 ,  0.08571473, -0.0229641 ,
        0.09599284, -0.07978653, -0.01027901, -0.04600375, -0.08503004,
       -0.11858507,  0.0896583 , -0.11214044,  0.10384996,  0.10468969,
       -0.04339964, -0.02280671,  0.11077163,  0.07171229,  0.02087731,
        0.01044465, -0.02046845,  0.03208542,  0.00974717, -0.05475277,
        0.09316711, -0.01551822, -0.03367957,  0.0002229 , -0.1181412 ,
       -0.08495402,  0.06369291,  0.0304447 , -0.07581333, -0.00241176,
        0.00259722, -0.00210012, -0.10811352, -0.01518592, -0.05603451,
       -0.0667677 ,  0.12343923,  0.02478511,  0.10393179, -0.10465047,
        0.07101591, -0.05284143, -0.07892408,  0.12068507,  0.00

In [15]:
forward_propagate(params, X[0:1], hidden_size, input_size, num_labels)

TypeError: can't multiply sequence by non-int of type 'float'

In [16]:
def cost(params, X, y, hidden_size, input_size, num_labels):
    a_0, z_1, a_1, z_2, y_pre = forward_propagate(params, X, hidden_size, input_size, num_labels)
    return np.sum(((y_pre - y) ** 2)) / 2

In [17]:
cost(params, X[0:1], y[0:1], hidden_size, input_size, num_labels)


TypeError: can't multiply sequence by non-int of type 'float'

In [18]:
def sigmoid_gradient(z):  # sigmoid导数
    return np.multiply(sigmoid(z), (1 - sigmoid(z)))

# 反向传播
- 标准BP算法
- 积累BP算法

In [19]:
def backprop_one(params, X, y, hidden_size, input_size, num_labels, learning_rate):
    theta = separate_params(params, hidden_size, input_size, num_labels)
    a_0, z_1, a_1, z_2, y_pre = forward_propagate(params, X, hidden_size, input_size, num_labels)
    z_1 = np.c_[z_1, 1]  # 这里为什么要加一列呢, 因为delta_1应该只有5列, 但是theta[1]有6列, 多一列是增加的全1列导致的
    gradient = [None, None]
    delta_2 = (y_pre - y) * y_pre * (1 - y_pre)
    gradient[1] = delta_2.T @ a_1
    delta_1 = delta_2 @ theta[1] * sigmoid_gradient(z_1)
    gradient[0] = delta_1[:, :-1].T @ a_0  # 这里再去掉一列
    return np.concatenate((np.ravel(gradient[0]), np.ravel(gradient[1]))) * learning_rate

In [20]:
learning_rate = 0.1
backprop_one(params, X[0:1], y[0:1], hidden_size, input_size, num_labels, learning_rate)

TypeError: can't multiply sequence by non-int of type 'float'

In [23]:
def backprop_all(params, X, y, hidden_size, input_size, num_labels, learning_rate):
    n = X.shape[0]
    gradient = create_params(hidden_size, input_size, num_labels)
    gradient -= gradient
    for i in range(n):
        temp = backprop_one(params, X[i:i + 1], y[i:i + 1], hidden_size, input_size, num_labels, learning_rate)
        gradient += temp
    c = cost(params, X, y, hidden_size, input_size, num_labels)
    return c, gradient / n

In [24]:
from scipy.optimize import minimize  # 优化函数
fmin = minimize(fun=backprop_all, x0=params, args=(X, y, hidden_size, input_size, num_labels, learning_rate),
                method='TNC', jac=True, options={'maxiter': 250}, )

fmin

/var/folders/b4/qwh98z_d37gg2_qygwhw2x900000gn/T/ipykernel_67738/1878526983.py:2: OptimizeWarning: Unknown solver options: maxiter
  fmin = minimize(fun=backprop_all, x0=params, args=(X, y, hidden_size, input_size, num_labels, learning_rate),


TypeError: can't multiply sequence by non-int of type 'float'